# 04 — Global ML Forecaster

This notebook builds a global machine-learning forecasting model.

Instead of fitting one model per clinic, the global model learns from all clinics at once. This is useful when many clinics share similar demand patterns and some clinics have limited history.


In [ ]:
from pathlib import Path
import sys

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

PROJECT_ROOT = Path.cwd().resolve()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent

SRC = PROJECT_ROOT / "src"
if str(SRC) not in sys.path:
    sys.path.insert(0, str(SRC))

pd.set_option("display.max_columns", 80)
plt.rcParams["figure.figsize"] = (11, 4)


In [ ]:
from clinic_forecast.data import generate_synthetic_healthcare_data
from clinic_forecast.features import make_supervised_frame
from clinic_forecast.metrics import compute_metrics, metrics_by_group
from clinic_forecast.models.ml import GlobalMLForecaster

data_path = PROJECT_ROOT / "data" / "raw" / "clinic_usage.csv"
if data_path.exists():
    usage = pd.read_csv(data_path, parse_dates=["date"])
else:
    usage, _, _ = generate_synthetic_healthcare_data()
    usage["date"] = pd.to_datetime(usage["date"])

cutoff = usage["date"].max() - pd.Timedelta(days=90)
train = usage[usage["date"] <= cutoff].copy()
test = usage[usage["date"] > cutoff].copy()
combined = pd.concat([train, test], ignore_index=True).sort_values(["clinic_id", "date"])


## Feature matrix

The ML model uses lag features, rolling means, calendar variables, clinic metadata and marketing variables.


In [ ]:
supervised = make_supervised_frame(combined)
supervised.head()


In [ ]:
model = GlobalMLForecaster()
model.fit(train)
predictions = model.predict_known_future(combined)
predictions = predictions[predictions["date"] > cutoff]

scored = test.merge(
    predictions[["clinic_id", "date", "forecast", "model"]],
    on=["clinic_id", "date"],
    how="inner",
    suffixes=("", "_pred"),
)

compute_metrics(scored["visits"], scored["forecast"])


## Error by clinic

A healthcare network needs clinic-level diagnostics. A good network average can hide poor forecasts in smaller clinics.


In [ ]:
clinic_metrics = metrics_by_group(scored, group_col="clinic_id", actual_col="visits", forecast_col="forecast")
clinic_metrics.sort_values("wape").head(12)


In [ ]:
clinic_id = clinic_metrics.sort_values("wape").iloc[0]["clinic_id"]
example = scored[scored["clinic_id"] == clinic_id]

fig, ax = plt.subplots()
ax.plot(example["date"], example["visits"], label="actual")
ax.plot(example["date"], example["forecast"], label="forecast")
ax.set_title(f"Global ML forecast — {clinic_id}")
ax.set_xlabel("Date")
ax.set_ylabel("Visits")
ax.legend()
plt.show()


## Optional XGBoost swap

The default implementation uses scikit-learn to keep the core PoC lightweight. In a production benchmark, XGBoost can be swapped in using the same feature matrix.


In [ ]:
try:
    from xgboost import XGBRegressor

    print("XGBoost is installed. You can replace the default estimator with XGBRegressor.")
    print(XGBRegressor)
except ImportError:
    print("XGBoost is not installed. Run `poetry install --with optional` to enable it.")
